# Recommender System From Scratch

Modern recommender systems look like one box on a homepage — "Because you watched X" — but inside is a four-stage pipeline: *candidate generation → feature assembly → ranking → re-ranking for business logic & freshness.* Each stage has its own model class, its own data contract, and its own evaluation metric. The hard parts are not the models themselves but the **joints between them**: how offline-trained candidate generators hand thousands of items to a real-time ranker without blowing the latency budget; how an offline-trained ranker is evaluated against logged feedback that was *produced by an older version of itself* (the off-policy problem); how a cold-start user with zero history gets useful recommendations; how an LLM can be slotted in as a re-ranker without becoming the slowest, most expensive component in the system.

This series builds a complete movie recommender from the bottom up — from the raw MovieLens 25M ratings file through a Flet desktop application that lets a real user rate movies, get recommendations, click, watch, and watch the metrics dashboard respond. Every module lives in `src/notebooks/recsys/` as an importable Python package, and every notebook in the series adds exactly one layer of the stack.


## About This Series

**Goal.** Build a [production-quality movie recommender]{.mark} in the style of Netflix/Spotify/YouTube, running end-to-end on one laptop. The recommender code lives in `src/notebooks/recsys/` as an importable Python package; every notebook adds one layer.

**Audience.** An ML practitioner who can train a model in PyTorch and serve a Python function, but has not stood up a full candidate-generation + ranking + feedback-loop pipeline before. Comfortable with linear algebra, embeddings, and the OpenAI API. Familiarity with Pydantic, FastAPI, Flet, and MLflow is helpful — those libraries are introduced as needed.

**Stack.** [PyTorch](https://pytorch.org) for models, [FAISS](https://faiss.ai) for ANN retrieval, [MLflow](https://mlflow.org) for the model registry, [FastAPI](https://fastapi.tiangola.com) for serving, [Flet](https://flet.dev) for the desktop UI, [OpenRouter](https://openrouter.ai) for the LLM re-ranker (default model `anthropic/claude-sonnet-4`). Secrets come from environment variables (`OPENROUTER_API_KEY`, `MLFLOW_TRACKING_URI`). The package is installed in editable mode via `uv sync` and any cell can `import notebooks.recsys` immediately.


## Course Notebooks

| # | Title | What We Build | Key Concepts |
|---|---|---|---|
| REC:01 | [Data & Biases](./01-data.html) | `load_movielens`, `Ratings` schema, time-based split, EDA | Explicit vs implicit feedback, position/popularity/selection bias, time-based vs random splits, sparsity |
| REC:02 | [Feature Store & Hashing](./02-features.html) | `FeatureStore`, `MovieFeatures`, `UserFeatures`, `hash_to_bucket` | Online/offline parity, feature hashing (Weinberger et al. 2009), feature transforms at serving time, Feast-lite |
| REC:03 | [Classic CF & Metrics](./03-classic.html) | ALS + content-based baselines; `Metricator` (NDCG@K, Recall@K, Coverage, Novelty) | Matrix factorization, ALS updates, content-based fallback, beyond-accuracy metrics |
| REC:04 | [Two-Tower Retrieval](./04-retrieval.html) | Embedding model + FAISS `ANNIndex`; sampled/in-batch negatives | Two-tower architecture, in-batch vs sampled softmax, retrival serving via ANN |
| REC:05 | [Ranking Models](./05-ranking.html) | Wide&Deep / DCN ranker; loss configuration | Cross+deep interactions (Wide&Deep), deep & cross network (DCN), pointwise/pairwise/listwise loss trade-offs |
| REC:06 | [Sequence & LLM Re-ranking](./06-sequence-llm.html) | SASRec + OpenRouter LLM re-ranker | Self-attention session models (SASRec), generative re-ranking, RAG embeddings from TMDB plot text |
| REC:07 | [Serving & Model Registry](./07-serving.html) | FastAPI `RecService` + MLflow registry + Feast-lite feature store | Offline batch + online endpoints, online/offline feature parity, model versioning, rollback |
| REC:08 | [Off-policy Eval & A/B](./08-eval.html) | `Evaluator`, IPS estimator, simulated A/B, ε-greedy bandit | Counterfactual evaluation, inverse propensity scoring, A/B test design, exploration/exploitation |
| REC:09 | [Demo App: End-to-End](./09-app.html) | `RecApp` Flet UI: onboard → rate → recs → click feedback → metrics dashboard | Live system, model toggle (classic → DL → LLM), feedback loop, drift / latency / coverage monitoring |

: {tbl-colwidths="[5,20,35,40]"}


## What You'll Build Understanding Of

- Why *time-based splits* beat random splits for recsys evaluation, and how popularity bias leaks future information unless you split carefully
- The four-stage system architecture (retrieval → feature fetch → ranking → re-ranking) and where the latency, data, and model drift costs actually live
- The two-tower + in-batch-negative training recipe and why it generalizes to billion-item catalogs without exploding softmax denominators
- The *loss selection* problem: when pointwise (regression), pairwise (BPR), and listwise (LambdaLoss) objectives give different orderings and why one is not strictly better
- Counterfactual evaluation: how an IPS estimator corrects the bias in logged data from the deploying policy, and where the variance kills you
- The cold-start triangle (new user, new item, new context) and which technique (content, LLM, contextual bandit) closes each gap
- How the same feature lookup ensures online/offline parity, and the operational cost of breaking it
- Integrating a live recsys service into a Flet desktop UI with streaming recommendations, click capture, and a real-time monitoring dashboard


## Prerequisites

- **Python 3.13+** with `uv` installed (`make venv` to create the environment)
- **PyTorch fluency:** comfortable writing a `nn.Module`, broadcasting, mini-batch loops, `torch.utils.data`
- **Linear algebra:** SVD, low-rank decompositions, dot-product geometry
- **Probability/statistics:** MAP estimation, importance sampling basics, A/B testing intuition
- **Data:** MovieLens 25M downloaded from [grouplens.org/datasets/movielens](https://grouplens.org/datasets/movielens/) and TMDB 5000; place both under `data/ml-25m/` and `data/tmdb/`
- **Environment:** `OPENROUTER_API_KEY` set in your shell for REC:06's live LLM re-ranker; `MLFLOW_TRACKING_URI` set so you can share the registry on a local server


## How to Read This Series

**Linear read.** Each notebook builds on the previous one's source files. Reading in order (01 → 09) gives the full architectural story: data → features → metrics → models → serving → eval → UI.

**Jump in.** The arc is split into three readable blocks:

- *Foundations (01–03):* data, features, metrics, classic baselines. Self-contained — read these first if new to recsys.
- *Modern models (04–06):* retrieval, ranking, sequence + LLM. Stages 01–03 are prerequisites; 04 is independent of 05 and 06.
- *Production (07–09):* serving, off-policy eval, demo app. Depends on the model package from 03–06.

**Use the package.** After `uv sync`, every module is importable: `from notebooks.recsys.serving import RecService`. The quickest way to run the demo app after reading REC:09:

```bash
uv run flet run src/notebooks/recsys/ui/app.py
```

**Run the serving backend.** From REC:07 onward, a FastAPI service exposes the recommender over HTTP:

```bash
uv run uvicorn notebooks.recsys.serving:app --port 8000 --reload
```
